# xHuBERT Experiment 5: Fusion Redundancy
**De tai**: He thong goi y san pham dua tren phan tich giong noi va cam xuc
**Hoc vien**: Nguyen Tan Nhu | **GVHD**: TS. Bui Thanh Hung (IUH)

**Pre-requisite**: Run Exp3 first (needs embeddings + checkpoints)

## Models
| Model | Description |
|-------|-------------|
| B0-control | FT-emb only (matched architecture) |
| F1-ConcatMLP | Concat(FT-emb, MFCC, Prosody) -> MLP |
| F2-AttGate | 3-branch attention gate |
| F2-AttGate-forced | AttGate + entropy regularization |
| F3-CrossAttn | Cross-attention fusion |
| B1-MFCC-only | MFCC only baseline |
| B2-Prosody-only | Prosody only baseline |

**Runtime**: **T4 GPU**

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("WARNING: GPU not enabled! Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

WORK = "/content/drive/MyDrive/xhubert_results"
os.makedirs(WORK, exist_ok=True)

os.environ["XHUBERT_SAVE_DIR"] = WORK
os.environ["RAVDESS_ROOT"] = os.path.join(WORK, "RAVDESS")

print(f"Working directory: {WORK}")

In [ ]:
!pip install -q --upgrade transformers==4.51.3 librosa huggingface-hub safetensors tqdm seaborn

import torch, transformers, librosa
print(f"torch:        {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"librosa:      {librosa.__version__}")

In [ ]:
import os, subprocess

REPO_DIR = "/content/ravdess_experiment"

if os.path.exists(REPO_DIR):
    print("Repo exists, pulling latest ...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print("Cloning repo ...")
    subprocess.run([
        "git", "clone", "-b", "feature/xhubert-rewrite",
        "https://github.com/nhunet/ravdess_experiment.git", REPO_DIR
    ], check=True)

os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

# Verify required files
required = [
    "config.py", "data.py", "features.py", "protocols.py",
    "stats.py", "utils.py",
    "models/__init__.py", "models/ml_classifiers.py",
    "models/xhubert.py", "models/hubert_vanilla.py",
    "models/fusion.py", "models/dl_1d.py", "models/dl_2d.py",
    "experiments/__init__.py",
]
missing = [f for f in required if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")
print("All required files OK")

In [ ]:
import os

DRIVE_RAVDESS = os.environ["RAVDESS_ROOT"]

if not os.path.exists(DRIVE_RAVDESS):
    print("Not in Drive -> Download from Zenodo...")
    !wget -q --show-progress https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip
    !unzip -q Audio_Speech_Actors_01-24.zip -d /content/RAVDESS_tmp
    !mkdir -p "$DRIVE_RAVDESS"
    !cp -r /content/RAVDESS_tmp/* "$DRIVE_RAVDESS/"
    !rm -rf /content/RAVDESS_tmp Audio_Speech_Actors_01-24.zip
    print("Saved to Drive.")
else:
    import glob
    n = len(glob.glob(os.path.join(DRIVE_RAVDESS, "**/*.wav"), recursive=True))
    print(f"Already exist ({n} wav files), don't download.")

In [ ]:
import config
from utils import ensure_dirs

ensure_dirs()
print(f"SAVE_DIR:  {config.SAVE_DIR}")
print(f"CSV_DIR:   {config.CSV_DIR}")
print(f"FIG_DIR:   {config.FIG_DIR}")
print(f"CKPT_DIR:  {config.CKPT_DIR}")
print(f"EMB_DIR:   {config.EMB_DIR}")
print(f"LOG_DIR:   {config.LOG_DIR}")
print(f"RAVDESS:   {config.RAVDESS_ROOT}")

### Keep Colab Alive
Paste this into your **browser Console** (F12 -> Console) to prevent idle timeout:
```javascript
function ClickConnect() {
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
```

## Verify Exp3 Outputs

In [ ]:
import os, glob, config
emb_files = glob.glob(os.path.join(config.EMB_DIR, "embeddings_trainval_fold*_seed42.npy"))
print(f"Found {len(emb_files)} embedding files from Exp3")
if len(emb_files) < 6:
    print("WARNING: Need 6 folds of embeddings. Run Exp3 LOSGO seed=42 first!")
else:
    print("All 6 folds available!")

## Run Fusion Experiment

In [ ]:
from data import RavdessDataset
from experiments.exp5_fusion import run_exp5
import config

dataset_hc = RavdessDataset(sr=config.SR_HANDCRAFTED)
df_exp5 = run_exp5(dataset_hc=dataset_hc, force=False)
print(df_exp5.groupby("Model")["accuracy"].agg(["mean", "std"]).round(2))

## TOST Equivalence Test

In [ ]:
import pandas as pd, numpy as np, os, config
from stats import tost_test

csv_path = os.path.join(config.CSV_DIR, "results_exp5_fusion.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    b0 = df[df["Model"] == "B0-control"].groupby("Fold")["accuracy"].mean().values

    for fusion_name in ["F1-ConcatMLP", "F2-AttGate", "F3-CrossAttn"]:
        fusion = df[df["Model"] == fusion_name].groupby("Fold")["accuracy"].mean().values
        if len(fusion) == len(b0) and len(b0) > 0:
            result = tost_test(fusion, b0, delta=2.0)
            print(f"{fusion_name} vs B0: TOST p={result['p_tost']:.4f}, "
                  f"equivalent={result['equivalent']}")
else:
    print("Run Exp5 first!")

## Gate Weights

In [ ]:
from visualization.plots import plot_exp5_gate_weights
import json, os, numpy as np, config

gate_weights = []
for fold in range(6):
    path = os.path.join(config.LOG_DIR, f"exp5_F2-AttGate_fold{fold}_seed42.json")
    if os.path.exists(path):
        with open(path) as f:
            data = json.load(f)
        if "gate_weights" in data:
            gate_weights.append(np.array(data["gate_weights"]))

if gate_weights:
    plot_exp5_gate_weights(gate_weights)
    print("Gate weights figure saved!")
else:
    print("No gate weight data found")